# LNCLIP-DF — Fast Training (frozen embeddings + attention head)

The CLIP ViT-L/14 backbone is frozen, so its per-frame features never change
across epochs. This notebook runs the backbone **once** to cache per-frame
embeddings, then trains a lightweight attention-pooling head on them.

- **Old path:** ~20 min/epoch (32 ViT-L/14 passes per video, every epoch, npz read from Drive).
- **This path:** cache once (~5 min), then **~1 s/epoch** — enough to run full 5-fold CV in minutes.

**What changes for AUC:** attention pooling over frames (not mean pooling),
explicit missing-face handling, embedding-space augmentation, flip-TTA, and an
honest **out-of-fold cross-validated AUC** instead of a single noisy test split.

**Prereq:** the pixel cache from your previous run (`cache_dual/*.npz`) must exist
on Drive. This notebook does NOT re-run face detection — it reuses that work.

In [ ]:
# CELL 1: Install + GPU
import torch
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
!pip install -q open_clip_torch scikit-learn tqdm pandas matplotlib
print('done')

In [ ]:
# CELL 2: Mount + clone + paths
from google.colab import drive
import os, sys, glob
from pathlib import Path
import pandas as pd

drive.mount('/content/drive')

if not os.path.exists('/content/eidon-deepfake'):
    !git clone https://github.com/rxbinsingh/eidon-deepfake /content/eidon-deepfake
else:
    !git -C /content/eidon-deepfake pull --quiet
os.chdir('/content/eidon-deepfake')
sys.path.insert(0, '/content/eidon-deepfake')

BASE       = '/content/drive/MyDrive/VIPER'
DATA_DIRS  = [f'{BASE}/dataset_production', f'{BASE}/dataset_week2', f'{BASE}/dfd_dataset']
PIXEL_CACHE = f'{BASE}/cache_dual'          # existing per-frame pixel cache (reused)
EMB_DRIVE  = f'{BASE}/emb_cache'            # embedding cache backup on Drive
EMB_LOCAL  = '/content/emb_cache'           # embedding cache used for training (fast local disk)
SAVE_DIR   = f'{BASE}/checkpoints_fast'
os.makedirs(EMB_DRIVE, exist_ok=True); os.makedirs(EMB_LOCAL, exist_ok=True); os.makedirs(SAVE_DIR, exist_ok=True)

print('Pixel cache files:', len(glob.glob(f'{PIXEL_CACHE}/*.npz')))
print('Embedding cache (Drive):', len(glob.glob(f'{EMB_DRIVE}/*.npz')))

In [ ]:
# CELL 3: Find all videos (same resolver as before)
import pandas as pd, os
from pathlib import Path

def get_all_videos():
    samples = []
    for d in DATA_DIRS:
        meta = f'{d}/metadata.csv'
        if not os.path.exists(meta): continue
        df = pd.read_csv(meta); found = 0
        for _, row in df.iterrows():
            label_str = row.get('label', 'unknown')
            label = 0 if label_str == 'real' else 1
            cat = row.get('category', label_str)
            fname = row['filename']
            paths = [f'{d}/{cat}/{fname}', f'{d}/{label_str}/{fname}']
            if 'original_path' in row and pd.notna(row['original_path']):
                paths += [f'{d}/{row["original_path"]}', f'{d}/{Path(row["original_path"]).parent}/{fname}']
            if 'dfd' in d.lower():
                paths += [f'{d}/DFD_original sequences/{fname}',
                          f'{d}/DFD_manipulated_sequences/{fname}',
                          f'{d}/DFD_manipulated_sequences/DFD_manipulated_sequences/{fname}']
            for vp in paths:
                if os.path.exists(vp):
                    samples.append((vp, label, cat)); found += 1; break
        print(f'  {Path(d).name}: {found}')
    return samples

all_videos = get_all_videos()
print(f'Total: {len(all_videos)} | Real: {sum(1 for _,l,_ in all_videos if l==0)} | Fake: {sum(1 for _,l,_ in all_videos if l==1)}')

In [ ]:
# CELL 4: Build embedding cache (ONCE). Reuses the pixel cache; skips existing.
import glob, shutil
from src.embed_cache import build_embedding_cache

# Restore any previously-built embeddings from Drive to local disk first
if len(glob.glob(f'{EMB_LOCAL}/*.npz')) < len(glob.glob(f'{EMB_DRIVE}/*.npz')):
    print('Restoring embedding cache from Drive -> local...')
    for f in glob.glob(f'{EMB_DRIVE}/*.npz'):
        shutil.copy(f, EMB_LOCAL)

need = len(all_videos) - len(glob.glob(f'{EMB_LOCAL}/*.npz'))
if need > 0:
    print(f'Building embeddings for ~{need} videos...')
    build_embedding_cache(all_videos, PIXEL_CACHE, EMB_LOCAL, device='cuda', include_flip=True)
    # back up to Drive for persistence across sessions
    print('Backing up embeddings local -> Drive...')
    for f in glob.glob(f'{EMB_LOCAL}/*.npz'):
        dst = f'{EMB_DRIVE}/{Path(f).name}'
        if not os.path.exists(dst): shutil.copy(f, dst)
else:
    print('Embedding cache complete — skipping build.')
print('Embeddings ready:', len(glob.glob(f'{EMB_LOCAL}/*.npz')))

In [ ]:
# CELL 5: Load embeddings + leakage-safe cross-validation
from pathlib import Path
from src.attn_head import EmbeddingBank
from src.grouping import derive_groups, leakage_report
from src import cv_train

stems = [(Path(vp).stem, cat) for vp, _l, cat in all_videos]
bank = EmbeddingBank(stems, EMB_LOCAL, device='cuda')

# Identity/source group per video so a source clip's real and its face-swaps
# (and DFD's cross-swapped actors) never land on opposite sides of a fold.
g_by_stem = {Path(vp).stem: g for (vp, _l, _c), g in zip(all_videos, derive_groups(all_videos))}
groups = [g_by_stem[s] for s in bank.stems]
present = [(s, int(bank.label[i]), bank.categories[i]) for i, s in enumerate(bank.stems)]
print('Leakage a RANDOM split would exploit:', leakage_report(present, groups))

print('\n########## Leakage-SAFE (grouped) CV — the honest number ##########')
res_grouped = cv_train.run_cv(bank, device='cuda', cfg=dict(n_splits=5), groups=groups, save_dir=SAVE_DIR)

print('\n########## Random (leaky) CV — for comparison only ##########')
res_leaky = cv_train.run_cv(bank, device='cuda', cfg=dict(n_splits=5), groups=None)

print(f"\nHonest grouped OOF AUC : {res_grouped['oof_auc']:.4f}")
print(f"Leaky random OOF AUC   : {res_leaky['oof_auc']:.4f}")
print(f"Leakage inflation      : {res_leaky['oof_auc'] - res_grouped['oof_auc']:+.4f}")


In [ ]:
# CELL 6: Leave-one-generator-out — does it catch UNSEEN deepfake methods?
# Reuses `bank` and `groups` from Cell 5 (run that first).
from src import logo_eval

logo = logo_eval.run_logo(
    bank, groups, device='cuda',
    families=logo_eval.DEFAULT_FAMILIES,  # faceswap / expression / t2v_generative
    save_dir=SAVE_DIR,
)


In [ ]:
# CELL 7: Regularization sweep — attack the train-OOF overfitting gap
# Reuses `bank` and `groups` from Cell 5. Runs grouped CV per config (~1-2 min each).
from src import reg_sweep

sweep = reg_sweep.run_sweep(bank, groups, device='cuda')

# To lock in the winner: copy its config into Cell 5 and re-run, e.g.
#   best = sweep[0]['config']
#   res = cv_train.run_cv(bank, device='cuda', groups=groups, save_dir=SAVE_DIR,
#       cfg=dict(dropout=best['dropout'], weight_decay=best['weight_decay'],
#                frame_drop=best['frame_drop'], noise_frac=best['noise_frac'],
#                hidden=best['hidden'], heads=best['heads'], n_splits=5))


In [ ]:
# CELL 8: Forensic stream — do frequency + noise features add anything?
# Builds a cacheable forensic descriptor per frame (once, CPU) and concatenates it
# onto the CLIP streams, then compares CLIP vs CLIP+forensic on BOTH honest AUC
# and unseen-generator generalization (the real test).
import glob, shutil, os
from pathlib import Path
from src.forensic_cache import build_forensic_cache
from src.attn_head import EmbeddingBank
from src.grouping import derive_groups
from src import cv_train, logo_eval

FOR_DRIVE = f'{BASE}/forensic_cache'; FOR_LOCAL = '/content/forensic_cache'
os.makedirs(FOR_DRIVE, exist_ok=True); os.makedirs(FOR_LOCAL, exist_ok=True)
if len(glob.glob(f'{FOR_LOCAL}/*.npz')) < len(glob.glob(f'{FOR_DRIVE}/*.npz')):
    for f in glob.glob(f'{FOR_DRIVE}/*.npz'): shutil.copy(f, FOR_LOCAL)

need = len(all_videos) - len(glob.glob(f'{FOR_LOCAL}/*.npz'))
if need > 0:
    print(f'Building forensic features for ~{need} videos (CPU, a few min)...')
    build_forensic_cache(all_videos, PIXEL_CACHE, FOR_LOCAL)
    for f in glob.glob(f'{FOR_LOCAL}/*.npz'):
        dst = f'{FOR_DRIVE}/{Path(f).name}'
        if not os.path.exists(dst): shutil.copy(f, dst)
print('Forensic features ready:', len(glob.glob(f'{FOR_LOCAL}/*.npz')))

# Best regularization config from the sweep (strong_all)
best_cfg = dict(dropout=0.5, weight_decay=0.05, frame_drop=0.25,
                noise_frac=0.2, hidden=128, heads=2, n_splits=5)

# CLIP + forensic bank
stems = [(Path(vp).stem, cat) for vp, _l, cat in all_videos]
bank_f = EmbeddingBank(stems, EMB_LOCAL, device='cuda', forensic_dir=FOR_LOCAL)
g_by_stem = {Path(vp).stem: g for (vp, _l, _c), g in zip(all_videos, derive_groups(all_videos))}
groups_f = [g_by_stem[s] for s in bank_f.stems]

print('\n### CLIP + forensic — leakage-safe grouped CV ###')
res_f = cv_train.run_cv(bank_f, device='cuda', groups=groups_f, cfg=best_cfg, save_dir=f'{SAVE_DIR}_forensic')
print('\n### CLIP + forensic — leave-one-generator-out ###')
logo_f = logo_eval.run_logo(bank_f, groups_f, device='cuda', families=logo_eval.DEFAULT_FAMILIES, cfg=best_cfg)

print(f"\n{'='*56}")
print(f"  CLIP-only      : grouped OOF 0.9428 | LOGO 0.789   (prior, same config)")
print(f"  CLIP+forensic  : grouped OOF {res_f['oof_auc']:.4f} | LOGO {logo_f['mean_auc']:.4f}")
print(f"{'='*56}")


## Inference (fold ensemble)

`run_cv` saved 5 head checkpoints (`head_fold*.pt`) to `SAVE_DIR`. For a new
video: build its embedding the same way (pixel cache -> `build_embedding_cache`),
load it into an `EmbeddingBank`, average `P(fake)` across the 5 `DeepfakeHead`
models with flip-TTA (`cv_train._predict(..., tta=True)`). Averaging folds is a
free accuracy/robustness gain over any single model.

**Next steps toward production (separate pass):** probability calibration
(temperature scaling — the 0.5 threshold is not calibrated), a leakage-safe
`groups` split, a robustness sweep across JPEG/compression levels, and a clean
inference module in `eval/`.